<!-- 학습 보강 셀 -->

# 11. ChromaDB 학습 흐름

이 노트북은 ChromaDB를 영구 저장 가능한 벡터 DB로 사용하는 예제입니다.
FAISS가 로컬 벡터 검색 라이브러리에 가깝다면, ChromaDB는 컬렉션 단위로 데이터를 관리하는 벡터 데이터베이스에 가깝습니다.

In [ ]:
# ChromaDB 벡터 스토어 예제에 필요한 패키지 설치
# - 패키지명은 llama-index-vector-stores-chroma 입니다. 기존의 점(.) 표기는 pip 패키지명으로 잘못된 형식입니다.
# !pip install chromadb llama-index-vector-stores-chroma llama-index-llms-ollama llama-index-embeddings-ollama

In [ ]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext, VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

In [ ]:
# LLM과 임베딩 모델 설정
llm = Ollama(
    model='gemma2:2b',
    temperature=0.5,
    request_timeout=120,
)

embed_model = OllamaEmbedding(
    model_name='nomic-embed-text',
)

In [ ]:
# 데이터 로드
documents = SimpleDirectoryReader('../NewData/pdf_sample2/').load_data()
print('읽어온 문서 수:', len(documents))

In [ ]:
# ChromaDB 영구 저장 클라이언트 생성
# - path에 지정한 디렉토리에 벡터 DB가 저장됩니다.
db = chromadb.PersistentClient(
    path='./chroma_db',
)

# 컬렉션 생성 또는 로드
# - 같은 이름의 컬렉션이 이미 있으면 기존 컬렉션을 재사용합니다.
chroma_collection = db.get_or_create_collection('quickstart_ollama')

<!-- 학습 보강 셀 -->

## 컬렉션을 사용하는 이유

ChromaDB의 컬렉션은 관련 벡터들을 묶는 단위입니다.
프로젝트별, 문서 종류별, 임베딩 모델별로 컬렉션을 나누면 데이터를 관리하고 비교하기 쉬워집니다.

In [ ]:
# ChromaDB를 LlamaIndex의 인덱싱 및 검색 파이프라인에 통합합니다.
vector_store = ChromaVectorStore(
    chroma_collection=chroma_collection,
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

<!-- 학습 보강 셀 -->

## 반복 실행 시 중복 데이터 주의

`get_or_create_collection`은 기존 컬렉션이 있으면 그대로 재사용합니다.
같은 노트북을 여러 번 실행하면 같은 문서가 중복으로 들어갈 수 있으므로, 실험을 처음부터 다시 하려면 컬렉션이나 `chroma_db` 디렉토리를 정리해야 합니다.

In [ ]:
# 인덱스 생성 및 데이터 임베딩
# - 같은 컬렉션에 반복 실행하면 중복 문서가 쌓일 수 있습니다.
# - 깨끗하게 다시 만들고 싶다면 chroma_db 디렉토리나 컬렉션을 삭제한 뒤 실행하세요.
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    embed_model=embed_model,
    show_progress=True,
)

<!-- 학습 보강 셀 -->

## ChromaDB 저장 후 다음 노트북과 연결되는 지점

이 셀에서 생성된 벡터는 `./chroma_db`에 저장됩니다.
따라서 12번 노트북은 원본 PDF를 다시 읽지 않고, 이 저장된 ChromaDB 컬렉션을 바로 열어 질의합니다.

---
### 메모리 인덱스 질의

In [ ]:
# 쿼리 엔진 생성
query_engine = index.as_query_engine(llm=llm)

In [ ]:
# 쿼리 실행
query = '이 논문에서 제안하는 모델의 장점은 무엇인가? 한글로 답변해줘'
response = query_engine.query(query)

print()
print('질문:', query)
print('답변:', response)